# 实践 15.1  系谱与基因组信息的模拟群体遗传评估

《统计遗传与智能育种学》第十五章 · 电子资源 `15_hiblup_virtual.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：15_hiblup_virtual.ipynb。实践模拟5个世代、320个个体和400个独立模拟标记，通过双亲逐位点孟德尔传递同时保留真实系谱与真实育种值。代码使用Henderson表格法构造A，用标记构造并尺度校正G，再构造H；前260个个体有表型，最后一代60个个体完全留出。最终比较PBLUP、GBLUP和SSGBLUP对留出代真实育种值的预测，并同时输出校准斜率。

运行检查：重启内核并运行全部单元。预期：重建三列系谱、PLINK转换输入、A/G/H关系矩阵及三种育种值预测；留出代表型不得参与固定效应、方差比或EBV求解。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1501；320个个体、400个模拟标记、5个世代；前260个个体有表型，最后60个个体为留出代；180个已分型个体，其中120个同时有表型。不依赖外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1501
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

# 5个世代：80个创始个体 + 4代各60个体；后代从上一代各选1个父本和母本
m = 400
sizes = [80, 60, 60, 60, 60]
n = sum(sizes)
starts = np.cumsum([0] + sizes)
sex = np.empty(n, dtype='<U1')
sire = np.full(n, -1, dtype=int)
dam = np.full(n, -1, dtype=int)
generation = np.empty(n, dtype=int)
for g, (lo, hi) in enumerate(zip(starts[:-1], starts[1:])):
    generation[lo:hi] = g
    labels = np.array(['M'] * ((hi-lo)//2) + ['F'] * ((hi-lo) - (hi-lo)//2))
    rng.shuffle(labels)
    sex[lo:hi] = labels
    if g > 0:
        prev = np.arange(starts[g-1], starts[g])
        males, females = prev[sex[prev] == 'M'], prev[sex[prev] == 'F']
        sire[lo:hi] = rng.choice(males, hi-lo, replace=True)
        dam[lo:hi] = rng.choice(females, hi-lo, replace=True)

# 教学简化：400个标记独立分离，没有染色体连锁/重组；每个后代从双亲各随机传递1个等位基因
p0 = rng.uniform(0.10, 0.50, m)
geno = np.empty((n, m), dtype=float)
geno[:sizes[0]] = rng.binomial(2, p0, (sizes[0], m))
for i in range(sizes[0], n):
    paternal = rng.binomial(1, geno[sire[i]] / 2.0)
    maternal = rng.binomial(1, geno[dam[i]] / 2.0)
    geno[i] = paternal + maternal
assert np.isin(geno, [0.0, 1.0, 2.0]).all()
print('population/markers/generations', n, m, len(sizes))
print('founders/last-generation', sizes[0], sizes[-1])

# Henderson表格法根据系谱构造加性亲缘关系矩阵A
A = np.zeros((n, n), dtype=float)
for i in range(n):
    if sire[i] < 0:
        A[i, i] = 1.0
    else:
        s, d = sire[i], dam[i]
        A[i, :i] = A[:i, i] = 0.5 * (A[s, :i] + A[d, :i])
        A[i, i] = 1.0 + 0.5 * A[s, d]
print('A diagonal mean', round(float(np.diag(A).mean()), 3),
      'mean inbreeding', round(float((np.diag(A)-1).mean()), 4))

# 生成数量性状，并把加性真实育种值（TBV）保留作为验证真值
beta = rng.normal(0, 1, m)
tbv0 = (geno - 2*p0) @ beta
beta = beta / tbv0.std()
tbv = (geno - 2*p0) @ beta
season = rng.choice(['S1', 'S2', 'S3'], n)
age = rng.normal(180, 12, n)
resid_sd = 1.15
y = (tbv + (sex == 'M')*0.60 + (season == 'S2')*0.45
     - (season == 'S3')*0.30 + 0.025*(age-180)
     + rng.normal(0, resid_sd, n))

# 前260个体有表型，最后一代60个体完全作为前瞻测试集
train = np.arange(260)
test = np.arange(260, 320)
# 从第140号个体开始分型：180个分型个体中120个有表型，60个为留出代
geno_idx = np.arange(140, 320)
geno_train = geno_idx[geno_idx < 260]
sigma_a2 = np.var(tbv[train], ddof=1)
sigma_e2 = resid_sd**2
print('phenotyped/test/genotyped/genotyped+phenotyped', len(train), len(test), len(geno_idx), len(geno_train))
print('known teaching variance ratio lambda=sigma_e2/sigma_a2', round(float(sigma_e2/sigma_a2), 3))

# 构造G，再与已分型个体的A22做尺度匹配和5%系谱混合
# 等位基因频率只由“已分型+有表型”的训练个体估计，避免借用留出代表型
p = geno[geno_train].mean(axis=0) / 2.0
W = geno[geno_idx] - 2*p
denom = 2*np.sum(p*(1-p))
G = W @ W.T / denom
A22 = A[np.ix_(geno_idx, geno_idx)]
off = ~np.eye(len(geno_idx), dtype=bool)
b_scale = ((np.diag(A22).mean() - A22[off].mean()) /
           (np.diag(G).mean() - G[off].mean()))
a_shift = A22[off].mean() - b_scale*G[off].mean()
G_adj = a_shift + b_scale*G
G_blend = 0.95*G_adj + 0.05*A22 + 1e-6*np.eye(len(geno_idx))

# 经典单步关系：H^-1=A^-1 + block(G^-1-A22^-1)
H_inv = np.linalg.inv(A)
H_inv[np.ix_(geno_idx, geno_idx)] += np.linalg.inv(G_blend) - np.linalg.inv(A22)
H = np.linalg.inv(H_inv)
print('mean diagonal A/G_scaled/H',
      *(round(float(v), 3) for v in [np.diag(A).mean(), np.diag(G_blend).mean(), np.diag(H).mean()]))

# 使用训练表型同时估计固定效应，再用已知方差比做核BLUP
# 这里固定效应不会用留出代的y，避免旧版Notebook中的测试集信息泄漏
fixed = pd.get_dummies(pd.DataFrame({'sex': sex, 'season': season}),
                       drop_first=True, dtype=float)
Xfix = np.column_stack([np.ones(n), fixed.to_numpy(), (age-180)/12])

def kernel_blup(K, obs, y_obs, X_obs, sigma_a2, sigma_e2):
    V = sigma_a2*K[np.ix_(obs, obs)] + sigma_e2*np.eye(len(obs))
    Vinv_X = np.linalg.solve(V, X_obs)
    Vinv_y = np.linalg.solve(V, y_obs)
    bhat = np.linalg.solve(X_obs.T@Vinv_X, X_obs.T@Vinv_y)
    alpha = np.linalg.solve(V, y_obs - X_obs@bhat)
    return sigma_a2*K[:, obs]@alpha

ebv_A = kernel_blup(A, train, y[train], Xfix[train], sigma_a2, sigma_e2)
ebv_H = kernel_blup(H, train, y[train], Xfix[train], sigma_a2, sigma_e2)
gpos = {animal: pos for pos, animal in enumerate(geno_idx)}
gtrain_local = np.array([gpos[i] for i in geno_train])
gtest_local = np.array([gpos[i] for i in test])
ebv_G = kernel_blup(G_blend, gtrain_local, y[geno_train], Xfix[geno_train], sigma_a2, sigma_e2)

pred = {
    'PBLUP_A': ebv_A[test],
    'GBLUP_G': ebv_G[gtest_local],
    'SSGBLUP_H': ebv_H[test],
}
for name, value in pred.items():
    accuracy = np.corrcoef(value, tbv[test])[0, 1]
    slope = np.polyfit(value, tbv[test], 1)[0]
    print(name, 'test_TBV_r', round(float(accuracy), 3),
          'TBV_on_EBV_slope', round(float(slope), 3))

# 生成与HIBLUP官方输入语义一致的教学文件
from pathlib import Path
outdir = Path('outputs_15_hiblup')
outdir.mkdir(exist_ok=True)
ids = np.array([f'ind{i:04d}' for i in range(n)])
trait_export = y.copy()
trait_export[test] = np.nan  # 留出代在HIBLUP表型文件中也保持缺失
phe = pd.DataFrame({'id': ids, 'sex': sex, 'season': season, 'age': age, 'trait': trait_export})
phe.to_csv(outdir/'demo.phe', sep='\t', index=False, na_rep='NA')
pedigree = pd.DataFrame({
    'id': ids,
    'father': np.where(sire < 0, 'NA', ids[np.maximum(sire, 0)]),
    'mother': np.where(dam < 0, 'NA', ids[np.maximum(dam, 0)]),
})
pedigree.to_csv(outdir/'demo.ped', sep='\t', index=False)

# PLINK PED/MAP使用独立文件名，不再与HIBLUP三列系谱demo.ped混用
mk = m  # 导出全部400个教学标记，与Python中构建G的信息范围一致
pd.DataFrame({0: 1, 1: [f'M{i+1}' for i in range(mk)], 2: 0,
              3: np.arange(1, mk+1)*10000}).to_csv(
    outdir/'geno.map', sep='\t', header=False, index=False)
plink_rows = []
for i in geno_idx:
    row = [ids[i], ids[i], '0', '0', '1' if sex[i] == 'M' else '2', '-9']
    for g0 in geno[i, :mk].astype(int):
        row.extend(['A','A'] if g0 == 2 else (['A','G'] if g0 == 1 else ['G','G']))
    plink_rows.append(row)
pd.DataFrame(plink_rows).to_csv(outdir/'geno.ped', sep=' ', header=False, index=False)
print('wrote', outdir/'demo.phe', 'columns', list(phe.columns))
print('wrote', outdir/'demo.ped', 'columns', list(pedigree.columns))
print('wrote PLINK conversion inputs', outdir/'geno.ped', outdir/'geno.map')
print('test phenotype missing in exported demo.phe', int(phe.trait.isna().sum()))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1501
population/markers/generations 320 400 5
founders/last-generation 80 60
A diagonal mean 1.008 mean inbreeding 0.0075
phenotyped/test/genotyped/genotyped+phenotyped 260 60 180 120
known teaching variance ratio lambda=sigma_e2/sigma_a2 1.346
mean diagonal A/G_scaled/H 1.008 1.013 1.01
PBLUP_A test_TBV_r 0.406 TBV_on_EBV_slope 0.97
GBLUP_G test_TBV_r 0.415 TBV_on_EBV_slope 0.904
SSGBLUP_H test_TBV_r 0.469 TBV_on_EBV_slope 1.028
wrote outputs_15_hiblup/demo.phe columns ['id', 'sex', 'season', 'age', 'trait']
wrote outputs_15_hiblup/demo.ped columns ['id', 'father', 'mother']
wrote PLINK conversion inputs outputs_15_hiblup/geno.ped outputs_15_hiblup/geno.map
test phenotype missing in exported demo.phe 60


结果解释：在最后一代完全留出的前提下，PBLUP、GBLUP和SSGBLUP对真实育种值的相关分别为0.406、0.415和0.469；TBV对EBV的回归斜率分别为0.970、0.904和1.028。这一模拟中GBLUP只有120个同时有基因型和表型的训练个体，而SSGBLUP还利用了其他有表型但未分型个体的系谱信息。三种方法的比较结果仅适用于本例数据及方差设定。

**PLINK转换与HIBLUP运行命令**（外部工具代码模板，本 notebook 不执行）

```bash
plink2 --ped outputs_15_hiblup/geno.ped --map outputs_15_hiblup/geno.map --make-bed --out outputs_15_hiblup/geno
./hiblup --single-trait --pheno outputs_15_hiblup/demo.phe --pheno-pos 5 --dcovar 2,3 --qcovar 4 --pedigree outputs_15_hiblup/demo.ped --bfile outputs_15_hiblup/geno --add --threads 8 --out outputs_15_hiblup/demo_ss
```

可修改练习：减少已分型个体；把resid_sd从1.15增至1.8；再随机留出60个个体，同步重设train、test及geno_train交集。比较三种设置下的测试集准确度与校准斜率，解释亲缘关系为什么会使随机划分的评估更乐观。

**HIBLUP 单性状 SSGBLUP 命令示例（见正文15.3节；需安装 HIBLUP，本 notebook 不执行）**

```bash
# 当前HIBLUP官方手册：单性状SSGBLUP示例的核心结构
./hiblup --single-trait \
  --pheno demo.phe --pheno-pos 8 \
  --dcovar 2,3 --qcovar 4,5 --rand 6,7 \
  --pedigree demo.ped --bfile demo \
  --add --dom --threads 32 --out demo
```